# 📘 Data Governance with SQL

This notebook demonstrates how to use SQL to build a simple employee management database. It emphasizes data governance principles such as referential integrity, data validation, and clean schema design.

We use Python to communicate with the SQLite database and to print results in a readable format. However, all data creation, retrieval, and manipulation are done through SQL. This setup allows us to focus on learning SQL while using Python as a convenient interface.

In [1]:
# We use sqlite3 to interact with the database 
# and pandas to format query results as tables
import sqlite3
import pandas as pd

## 🧱 1. Initialize a SQLite Database

SQLite is a lightweight and self-contained relational database engine. Unlike traditional database systems that require a separate server process, SQLite stores all data in a single file on disk and runs directly within the application that accesses it. Because of its simplicity and zero-configuration setup, SQLite is widely used in embedded systems, mobile apps, and teaching environments.

In this section, we create an in-memory SQLite database and set up a cursor object to execute SQL commands. This temporary database exists only while the notebook is running, making it ideal for experimentation and instruction.

In [2]:
# Connect to an in-memory SQLite database
con = sqlite3.connect(':memory:')

# Create a cursor object to execute SQL commands
cur = con.cursor()

## 🛠️ 2. Define SQL Execution Helper Functions

This section defines reusable helper functions that let us execute SQL scripts and queries from files or strings. You don't need to modify or deeply understand this code. These are just functions to run SQL queries or scripts throughout the notebook.

In [3]:
# Define a helper function to execute a SQL script inside a file
def execute_sql_script_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            sql_script = f.read()

        print(f'🖥 Executing SQL script from file {filepath}')
        cur.executescript(sql_script)
        print(f"✅ Successfully executed SQL script from file {filepath}")
    except Exception as e:
        print(f"❌ An error occurred: {e}")


# Define a helper function to execute a SQL query string
def execute_sql_query(query):
    try:
        # Execute the query
        cur.execute(query)

        if cur.description is not None:
            # Fetch all rows and column names
            rows = cur.fetchall()
            column_names = [description[0] for description in cur.description]
    
            if rows:
                print(f"✅ Query executed successfully. {len(rows)} row{'s' if len(rows) >= 2 else ''} to display.")
            else:
                print("✅ Query executed successfully. No results to display.")
    
            df = pd.DataFrame(rows, columns=column_names)
            display(df.style.hide(axis='index'))
        else:
            # For INSERT, UPDATE, DELETE, etc.
            print("✅ Query executed successfully. No results to display.")
    except sqlite3.Error as e:
        print(f"❌ An error occurred: {e}")

# Define a helper function to execute a SQL query inside a file
def execute_sql_query_from_file(filepath):
        with open(filepath, 'r') as f:
            query_string = f.read()

            print(f'🖥 Executing SQL query from file {filepath}')
            execute_sql_query(query_string)


## 🗂️ 3. Create and Populate Database Tables

### 3.1 Create tables using SQL scripts

The SQL statements that define our database schema are stored in a **separate `.sql` script file**. This file contains the full `CREATE TABLE` commands for `departments`, `job_titles`, and `employees`, including all primary keys, foreign keys, and check constraints. In this notebook, we simply execute that script using a helper function. If you'd like to view, edit, or study the SQL in detail, you can open the file named `01_create_tables.sql` directly in your Jupyter notebook environment or a code editor.

In [4]:
execute_sql_script_from_file('./sql-scripts/01_create_tables.sql')

🖥 Executing SQL script from file ./sql-scripts/01_create_tables.sql
✅ Successfully executed SQL script from file ./sql-scripts/01_create_tables.sql


### 3.2 Populate tables with sample rows using SQL scripts

In [5]:
execute_sql_script_from_file('./sql-scripts/02_populate_tables.sql')

🖥 Executing SQL script from file ./sql-scripts/02_populate_tables.sql
✅ Successfully executed SQL script from file ./sql-scripts/02_populate_tables.sql


### 3.3 Print (query) all tables within the database

In [6]:
execute_sql_query('SELECT type, tbl_name FROM sqlite_master WHERE type="table";')

✅ Query executed successfully. 3 rows to display.


type,tbl_name
table,departments
table,job_titles
table,employees


### 3.4 Query all departments

In [7]:
execute_sql_query('SELECT * FROM departments;')

✅ Query executed successfully. 7 rows to display.


department_id,department_name
1,Engineering
2,Human Resources
3,Marketing
4,Finance
5,Operations
6,Sales
7,Legal


### 3.4 Query all job titles

In [8]:
execute_sql_query('SELECT * FROM job_titles;')

✅ Query executed successfully. 8 rows to display.


job_title_id,job_title
1,Software Engineer
2,HR Specialist
3,Marketing Manager
4,Data Analyst
5,Accountant
6,Operations Associate
7,Sales Executive
8,Legal Counsel


### 3.4 Query first 10 rows in the employees table

In [9]:
execute_sql_query('SELECT * FROM employees LIMIT 10;')

✅ Query executed successfully. 10 rows to display.


employee_id,full_name,start_date,end_date,department_id,job_title_id,status,pay_type,pay_rate
101,Allison Hill,2022-06-05,None,2,6,Active,Salary,61176.480000
102,Megan Mcclain,2021-04-08,None,6,8,Active,Salary,104280.490000
103,Brandon Hall,2020-10-02,None,6,5,Active,Salary,73851.090000
104,Matthew Gardner,2022-05-27,2024-11-27,5,7,Terminated,Salary,103833.130000
105,Sean Blake,2021-11-20,None,2,8,Active,Salary,98396.600000
106,Donald Lewis,2023-05-23,None,1,3,Active,Salary,99603.970000
107,Brent Abbott,2022-10-11,None,4,2,On-boarding,Salary,67876.650000
108,Kimberly Dudley,2023-10-01,None,4,5,Active,Hourly,54.430000
109,Sandra Montgomery,2023-08-09,2024-01-05,1,2,Terminated,Salary,67431.290000
110,Amber Perez,2022-12-28,None,3,6,Active,Salary,59203.820000


## 🔍 4. Using SQL to Retrieve Information



### 4.1 Find all on-boarding employees

In [10]:
execute_sql_query_from_file('./sql-scripts/03_all_onboarding_employees.sql')

🖥 Executing SQL query from file ./sql-scripts/03_all_onboarding_employees.sql
✅ Query executed successfully. 5 rows to display.


full_name,start_date,department_name,job_title,pay_type,pay_rate,status
Brent Abbott,2022-10-11,Finance,HR Specialist,Salary,67876.650000,On-boarding
Joyce Hickman,2021-02-22,Engineering,HR Specialist,Salary,65176.540000,On-boarding
Robert Stevens,2023-02-15,Human Resources,Marketing Manager,Hourly,52.430000,On-boarding
Jose Schultz,2023-05-19,Operations,Data Analyst,Salary,58148.820000,On-boarding
Michael Mccarthy,2021-01-27,Engineering,Software Engineer,Salary,104418.670000,On-boarding


### 4.2. Find the number of employees by status

In [11]:
execute_sql_query_from_file('./sql-scripts/04_count_of_employees_by_status.sql')

🖥 Executing SQL query from file ./sql-scripts/04_count_of_employees_by_status.sql
✅ Query executed successfully. 3 rows to display.


status,num_employees
Active,80
On-boarding,5
Terminated,15


### 4.3 Calculate the average salary by department

In [12]:
execute_sql_query_from_file('./sql-scripts/05_average_salary_by_department.sql')

🖥 Executing SQL query from file ./sql-scripts/05_average_salary_by_department.sql
✅ Query executed successfully. 7 rows to display.


department_name,num_employees,avg_salary
Engineering,14,83050.220000
Finance,6,82038.710000
Human Resources,12,82144.100000
Legal,13,81817.150000
Marketing,10,86951.650000
Operations,10,89158.030000
Sales,19,80967.250000


### 4.4 Find employees hired during or after 2025

In [13]:
execute_sql_query_from_file('./sql-scripts/06_employees_hired_after_2025.sql')

🖥 Executing SQL query from file ./sql-scripts/06_employees_hired_after_2025.sql
✅ Query executed successfully. 13 rows to display.


full_name,start_date,department_name
Edward Novak,2025-06-15,Human Resources
Lauren Dominguez,2025-06-04,Operations
John Ryan,2025-05-21,Operations
Megan Orr,2025-05-15,Engineering
Jennifer Jones,2025-05-09,Legal
David Garcia,2025-04-16,Sales
James Mayo,2025-04-07,Sales
Danny Morgan,2025-03-28,Legal
Deborah Preston,2025-03-26,Sales
Deborah Figueroa,2025-03-22,Legal


## 🔒 5. Using SQL to Impose Data Integrity

💡 **Note 1:** The queries in this section are **intentionally designed to fail**. They attempt to insert or update data in ways that violate the integrity rules we've defined in our schema - such as primary key constraints, foreign key references, and check conditions. These errors are a feature, not a bug: they demonstrate how well-structured databases protect themselves from invalid or inconsistent data. Understanding why these queries fail is just as important as writing queries that succeed.

**📝 Note 2:** In this section, we will use **multiline strings** in Python by wrapping our SQL queries with triple single quotes (`'''`). Triple single quotes (`'''`) or triple double quotes (`"""`) are used in Python to define **multiline string literals**. These allow you to span text across multiple lines without needing escape characters like `\n`. This is especially useful when writing SQL queries or large blocks of text.

### 5.1 Prevent duplicate `department_id`

Try executing an `INSERT` statement with a primary key that already exists in the `departments` table.

```python
execute_sql_query('''
    INSERT INTO departments (department_id, department_name) 
    VALUES (1, 'Sales');
''')
```

In [14]:
execute_sql_query('''
    INSERT INTO departments (department_id, department_name) 
    VALUES (1, 'Sales'); -- ⚠️ department_id == 1 already exists in the departments table
''')

❌ An error occurred: UNIQUE constraint failed: departments.department_id


🔍 Why it fails: `department_id = 1` already exists. This is a primary key constraint violation.

#### Correct Query

Note: Running this query multiple times will keep adding an extra row to the `departments` table with duplicate `department_name` value (`'Sales'`). This is because we have not added a `UNIQUE` constraint to the `department_name` column.

In [15]:
execute_sql_query('''
    INSERT INTO departments (department_name) 
    VALUES ('Sales');
''')

✅ Query executed successfully. No results to display.


Check the result.

In [16]:
execute_sql_query('''
    SELECT * FROM departments;
''')

✅ Query executed successfully. 8 rows to display.


department_id,department_name
1,Engineering
2,Human Resources
3,Marketing
4,Finance
5,Operations
6,Sales
7,Legal
8,Sales


### 5.2 Prevent a duplicate `job_title` value

Try executing an `INSERT` statement with a job title that already exists in the `job_titles` table.

```sql
execute_sql_query('''
    INSERT INTO job_titles (job_title) 
    VALUES ('Software Engineer'); -- ⚠️ `Software Engineer` already exists in the 
''')
```

In [17]:
execute_sql_query('''
    INSERT INTO job_titles (job_title) 
    VALUES ('Software Engineer'); -- ⚠️ `Software Engineer` already exists in the 
''')

❌ An error occurred: UNIQUE constraint failed: job_titles.job_title


🔍 Why it fails: `job_title == 'Software Engineer'` already exists in the `job_title` table. → `UNIQUE` constraint violation on `job_title`.

### 5.3 Prevent a negative `pay_rate` value

In [18]:
execute_sql_query('''
    INSERT INTO employees (
        employee_id,
        full_name,
        start_date,
        end_date,
        department_id,
        job_title_id,
        status,
        pay_type,
        pay_rate
    ) VALUES (
        108,
        'Ashley Potts',
        '2023-01-01',
        NULL,
        1,
        1,
        'Active',
        'Hourly',
        -25.00 -- ⚠️ pay_rate is a negative amount
    );
''')

❌ An error occurred: CHECK constraint failed: pay_rate > 0


🔍 Why it fails: `pay_rate` < 0 violates the `CHECK` constraint (`pay_rate` > 0)

### 5.4 Prevent an `end_date` that is earlier than the `start_date`

In [19]:
execute_sql_query('''
    INSERT INTO employees (
        employee_id,
        full_name,
        start_date,
        end_date,
        department_id,
        job_title_id,
        status,
        pay_type,
        pay_rate
    ) VALUES (
        109,
        'Jenny Park',
        '2024-01-01',
        '2022-12-01', -- ⚠️ end_date is 2022, while the start date is 2024
        1,
        1,
        'Terminated',
        'Salary',
        80000
    );
''')

❌ An error occurred: CHECK constraint failed: end_date IS NULL OR end_date >= start_date


🔍 Why it fails: `end_date` < `start_date` → `CHECK` constraint violation (`end_date IS NULL OR end_date >= start_date`)

### 5.5 Prevent an invalid employee `status` value

In [20]:
execute_sql_query('''
    INSERT INTO employees (
        employee_id,
        full_name,
        start_date,
        end_date,
        department_id,
        job_title_id,
        status,
        pay_type,
        pay_rate
    ) VALUES (
        107,
        'Ari Moss',
        '2023-09-01',
        NULL,
        1,
        1,
        'Actiive', -- ⚠️ Invalid status (typo) - should be 'Active' instead of 'Actiive'
        'Salary',
        75000
    );
''')

❌ An error occurred: CHECK constraint failed: status IN ('Active', 'On-boarding', 'Terminated')


🔍 Why it fails: 'Actiive' is not one of the allowed values according to the `CHECK` constraint on `status` (`IN ('Active', 'On-boarding', 'Terminated')`)

## ✅ 6. Summary and Next Steps
In this notebook, we built a simple SQLite database to demonstrate key data governance principles: unique identifiers, referential integrity, and validation rules through constraints. These foundations are essential in building systems that are reliable, auditable, and secure. In the real-world, you may extend this database with more advanced concepts such as triggers, views, and real-time data validation.